# 94b — Group nodal-only events into candidate shots (SAFE)

This notebook handles only the two T1 nodal-only surveys. It does **not** alter SQLite. It reads the completed catalog in immutable mode and writes provisional CSV/PNG review products to a separate directory.

The workflow is deliberately asymmetric:

1. **2026-05-17:** identify the ordered 104–140 m walk using a constrained, monotonic segmentation. Source estimates are evidence, not truth; temporal order supplies the one-metre staircase. The search is extended to 19:30 UTC in continuous N2 data.
2. **2026-05-19:** use Glenn's field times and known positions. Existing catalog detections are used for 110–114, 122, 130 and 10 m. Targeted end-receiver detection searches the continuous SDS archive for the missing 36, 216 and 290 m off-end stacks.

Nothing produced here is authoritative until the review figures and candidate counts are accepted. A later notebook can promote approved groups and extract/stack recovered continuous-data events.

In [ ]:
from pathlib import Path
import sqlite3
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import hilbert, find_peaks
from obspy import UTCDateTime
from obspy.clients.filesystem.sds import Client as SDSClient

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
CATALOG_DB = PROJECT_ROOT / 'catalog' / 'lbssp_shot_catalog.sqlite'
SDS_ROOT = PROJECT_ROOT / 'nodal_sds_position_codes'
GLENN_WORKBOOK = Path('/Users/thompsong/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP/04_FieldData/glenn_smartsolo_nodal_metadata_with_estimated_coords.xlsx')
OUT_ROOT = PROJECT_ROOT / 'nodal_only_candidate_review'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MAY17_CATALOG_START = pd.Timestamp('2026-05-17T17:10:00Z')
MAY17_CATALOG_END = pd.Timestamp('2026-05-17T19:00:00Z')
MAY17_EXTENSION_END = pd.Timestamp('2026-05-17T19:30:00Z')
MAY17_WALK_END = pd.Timestamp('2026-05-17T19:12:00Z')  # before the documented ~19:15 treefall
MAY17_POSITIONS_M = np.arange(104, 141, dtype=float)
MAY17_MIN_GROUP_EVENTS = 5
MAY17_MAX_GROUP_EVENTS = 45
MAY17_POSITION_OUTLIER_M = 8.0
DUPLICATE_TRIGGER_S = 0.80

# Targeted detector: deliberately local, unlike notebook 90's >=10-node coincidence trigger.
RECOVERY_FILTER_HZ = (5.0, 150.0)
RECOVERY_ENVELOPE_SMOOTH_S = 0.020
RECOVERY_MIN_SPACING_S = 1.50
RECOVERY_N_END_RECEIVERS = 8
RECOVERY_N_CENTER_RECEIVERS = 12

COLORS = {
    'catalog': '#0072B2', 'recovered': '#009E73', 'rejected': '#D55E00',
    'truth': '#111111', 'uncertain': '#CC79A7',
}

print('Read-only catalog:', CATALOG_DB)
print('Continuous SDS:', SDS_ROOT)
print('Review output:', OUT_ROOT)
print('May 17 search:', MAY17_CATALOG_START, 'to', MAY17_EXTENSION_END)

## 1. Load unassigned catalog events and field metadata

In [ ]:
uri = f'file:{CATALOG_DB}?mode=ro&immutable=1'
with sqlite3.connect(uri, uri=True) as conn:
    event_catalog = pd.read_sql(
        "SELECT * FROM nodal_event_catalog_qc WHERE final_event_status='unassigned'", conn
    )

event_catalog['event_time'] = pd.to_datetime(
    event_catalog['nodal_event_time_utc'], utc=True, errors='coerce', format='mixed'
)
event_catalog['estimated_source_x_m'] = pd.to_numeric(
    event_catalog['estimated_source_x_m'], errors='coerce'
)

may17_metadata = pd.read_excel(GLENN_WORKBOOK, sheet_name='T1_N2_Refraction')
may19_n2_metadata = may17_metadata.loc[may17_metadata['shot_no'].between(38, 42)].copy()
may19_n3_metadata = pd.read_excel(GLENN_WORKBOOK, sheet_name='T1_N3_Refraction')

may17_all_window = event_catalog.loc[
    event_catalog.nodal_timewindow_label.eq('T1_N2_Nodal1')
    & event_catalog.event_time.between(pd.Timestamp('2026-05-17T16:00:00Z'), MAY17_CATALOG_END, inclusive='left')
].copy()
may17_catalog = may17_all_window.loc[may17_all_window.event_time >= MAY17_CATALOG_START].copy()
may19_catalog = event_catalog.loc[
    event_catalog.nodal_timewindow_label.isin(['T1_N2_Nodal2', 'T1_N3_Nodal3'])
    & event_catalog.event_time.between(pd.Timestamp('2026-05-19T12:50:00Z'), pd.Timestamp('2026-05-19T14:20:00Z'), inclusive='both')
].copy()

print(f'May 17 full original window: {len(may17_all_window):,} unassigned detections')
print(f'May 17 provisional forward-walk window: {len(may17_catalog):,}')
print(f'May 19 catalog detections in review window: {len(may19_catalog):,}')
display(may19_n2_metadata[['shot_no', 'source_position_m', 'n_blows', 'notes']])
display(may19_n3_metadata[['shot_no', 'source_position_m', 'n_blows']])

## 2. Targeted continuous-data recovery

The original detector required coincidence across at least ten nodes. Here the detector uses a robust envelope score over the eight receivers nearest an appropriate line end. For the May 17 extension it uses twelve receivers nearest 138 m. Candidate peaks remain provisional and carry their recovery score.

In [ ]:
sds = SDSClient(str(SDS_ROOT))

def station_x_m(trace):
    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan

def choose_detection_traces(stream, selector, target_x=None):
    traces = sorted(stream, key=station_x_m)
    if selector == 'low':
        return traces[:RECOVERY_N_END_RECEIVERS]
    if selector == 'high':
        return traces[-RECOVERY_N_END_RECEIVERS:]
    if selector == 'center':
        return sorted(traces, key=lambda tr: abs(station_x_m(tr) - float(target_x)))[:RECOVERY_N_CENTER_RECEIVERS]
    raise ValueError(selector)

def robust_detection_arrays(location, start, end, selector, target_x=None):
    start_utc, end_utc = UTCDateTime(start), UTCDateTime(end)
    stream = sds.get_waveforms('T1', '*', location, 'DPZ', start_utc, end_utc)
    stream.merge(method=1, fill_value='interpolate')
    traces = choose_detection_traces(stream, selector, target_x=target_x)
    if not traces:
        raise RuntimeError(f'No DPZ traces for T1.*.{location}.DPZ {start} to {end}')
    fs = float(traces[0].stats.sampling_rate)
    standardized, filtered = [], []
    for original in traces:
        tr = original.copy()
        tr.data = tr.data.astype(np.float64)
        tr.detrend('linear')
        tr.taper(max_percentage=.02, type='hann')
        high = min(RECOVERY_FILTER_HZ[1], .45 * fs)
        tr.filter('bandpass', freqmin=RECOVERY_FILTER_HZ[0], freqmax=high, corners=4, zerophase=True)
        envelope = np.abs(hilbert(tr.data))
        n_smooth = max(1, int(round(RECOVERY_ENVELOPE_SMOOTH_S * fs)))
        envelope = np.convolve(envelope, np.ones(n_smooth) / n_smooth, mode='same')
        baseline = np.median(envelope)
        scale = 1.4826 * np.median(np.abs(envelope - baseline))
        standardized.append((envelope - baseline) / (scale + 1e-12))
        filtered.append(tr)
    n = min(map(len, standardized))
    score = np.quantile(np.vstack([x[:n] for x in standardized]), .65, axis=0)
    return start_utc, fs, score, filtered

def energy_centroid_at_peak(filtered_traces, peak_index, fs):
    i0 = max(0, peak_index - int(.05 * fs))
    i1 = peak_index + int(.35 * fs)
    rows = []
    for tr in filtered_traces:
        segment = tr.data[i0:min(i1, len(tr.data))]
        if len(segment):
            rows.append((station_x_m(tr), float(np.sum(segment.astype(float) ** 2))))
    rows = sorted(rows, key=lambda item: item[1], reverse=True)[:3]
    if not rows or sum(e for _, e in rows) <= 0:
        return np.nan
    return float(sum(x * e for x, e in rows) / sum(e for _, e in rows))

def targeted_recovery(spec):
    start, fs, score, traces = robust_detection_arrays(
        spec['location'], spec['start'], spec['end'], spec['selector'], spec.get('target_x')
    )
    peaks, props = find_peaks(
        score, height=spec['height'], prominence=spec.get('prominence', 3.0),
        distance=max(1, int(round(RECOVERY_MIN_SPACING_S * fs))),
    )
    rows = []
    for sequence, peak in enumerate(peaks, 1):
        event_time = start + peak / fs
        rows.append({
            'nodal_event_id': f"RECOV_{spec['group_key']}_{sequence:04d}",
            'event_time': pd.Timestamp(str(event_time)),
            'nodal_event_time_utc': str(event_time),
            'estimated_source_x_m': energy_centroid_at_peak(traces, peak, fs),
            'candidate_origin': 'targeted_continuous_sds',
            'recovery_group_key': spec['group_key'],
            'recovery_score': float(score[peak]),
            'assigned_source_x_m': spec.get('assigned_source_x_m'),
            'assigned_source_label': spec.get('assigned_source_label'),
            'position_basis': spec.get('position_basis', 'field_metadata'),
            'expected_blows': spec.get('expected_blows'),
        })
    return pd.DataFrame(rows), {'spec': spec, 'start': start, 'fs': fs, 'score': score, 'peaks': peaks}


In [ ]:
RECOVERY_SPECS = [
    {
        'group_key': 'MAY17_EXTENSION', 'location': 'N2', 'start': '2026-05-17T19:00:00Z',
        'end': '2026-05-17T19:30:00Z', 'selector': 'center', 'target_x': 138.0,
        'height': 10.0, 'position_basis': 'ordered_walk_extension',
    },
    {
        'group_key': 'MAY19_036M', 'location': 'N2', 'start': '2026-05-19T12:58:30Z',
        'end': '2026-05-19T13:04:10Z', 'selector': 'low', 'height': 15.0,
        'assigned_source_x_m': 36.0, 'assigned_source_label': '36m', 'expected_blows': 22,
    },
    {
        'group_key': 'MAY19_216M', 'location': 'N2', 'start': '2026-05-19T13:17:00Z',
        'end': '2026-05-19T13:20:10Z', 'selector': 'high', 'height': 5.0,
        'assigned_source_x_m': 216.0, 'assigned_source_label': '216m', 'expected_blows': 30,
    },
    {
        'group_key': 'MAY19_290M', 'location': 'N3', 'start': '2026-05-19T14:09:00Z',
        'end': '2026-05-19T14:13:00Z', 'selector': 'high', 'height': 5.0,
        'assigned_source_x_m': 290.0, 'assigned_source_label': '290m', 'expected_blows': 60,
    },
]

recovery_frames, recovery_diagnostics = [], []
for spec in RECOVERY_SPECS:
    print('Recovering', spec['group_key'], spec['start'], 'to', spec['end'])
    recovered, diagnostic = targeted_recovery(spec)
    recovery_frames.append(recovered)
    recovery_diagnostics.append(diagnostic)
    print('  candidates:', len(recovered))
recovered_events = pd.concat(recovery_frames, ignore_index=True)
recovered_events.to_csv(OUT_ROOT / '94b_targeted_recovered_events.csv', index=False)
display(recovered_events.groupby('recovery_group_key').agg(
    n_candidates=('nodal_event_id', 'size'), first=('event_time', 'min'), last=('event_time', 'max'),
    median_score=('recovery_score', 'median'), median_energy_x=('estimated_source_x_m', 'median')
).reset_index())

In [ ]:
fig, axes = plt.subplots(len(recovery_diagnostics), 1, figsize=(15, 3.2 * len(recovery_diagnostics)), constrained_layout=True)
for ax, diagnostic in zip(np.atleast_1d(axes), recovery_diagnostics):
    spec, start, fs, score, peaks = diagnostic['spec'], diagnostic['start'], diagnostic['fs'], diagnostic['score'], diagnostic['peaks']
    step = max(1, int(fs / 20))
    times = pd.to_datetime([str(start + i / fs) for i in range(0, len(score), step)], utc=True, format='mixed')
    ax.plot(times, score[::step], color='#777777', lw=.65, label='Robust end/center-receiver score')
    peak_times = pd.to_datetime([str(start + int(i) / fs) for i in peaks], utc=True, format='mixed')
    ax.scatter(peak_times, score[peaks], s=16, color=COLORS['recovered'], label=f'Recovered candidates (n={len(peaks)})')
    ax.axhline(spec['height'], color=COLORS['rejected'], ls='--', lw=1, label=f"Threshold {spec['height']:.1f}")
    ax.set(title=spec['group_key'], ylabel='Robust score')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.grid(alpha=.2)
    ax.legend(fontsize=8, loc='upper right')
axes[-1].set_xlabel('UTC time')
fig.suptitle('Targeted continuous-data recovery for missed nodal-only shots', fontsize=15)
fig.savefig(OUT_ROOT / '94b_targeted_recovery_overview.png', dpi=180, bbox_inches='tight')
plt.close(fig)


## 3. May 19 metadata-timed assignments

The sinkhole-middle source is retained as an uncertain 110–114 m location with 112 m used only as a plotting coordinate. The detailed refraction sheet, rather than the inconsistent special-events row, controls the assignment.

In [ ]:
MAY19_CATALOG_GROUPS = [
    {'group_key': 'MAY19_SINKHOLE_110_114M', 'start': '2026-05-19T13:07:20Z', 'end': '2026-05-19T13:09:40Z',
     'assigned_source_x_m': 112.0, 'assigned_source_label': 'sinkhole_middle_110_to_114m', 'expected_blows': 20, 'position_basis': 'field_metadata_uncertain_110_to_114m'},
    {'group_key': 'MAY19_122M', 'start': '2026-05-19T13:09:40Z', 'end': '2026-05-19T13:12:20Z',
     'assigned_source_x_m': 122.0, 'assigned_source_label': '122m', 'expected_blows': 32, 'position_basis': 'field_metadata'},
    {'group_key': 'MAY19_130M', 'start': '2026-05-19T13:13:30Z', 'end': '2026-05-19T13:16:00Z',
     'assigned_source_x_m': 130.0, 'assigned_source_label': '130m', 'expected_blows': 30, 'position_basis': 'field_metadata'},
    {'group_key': 'MAY19_010M', 'start': '2026-05-19T13:59:30Z', 'end': '2026-05-19T14:04:00Z',
     'assigned_source_x_m': 10.0, 'assigned_source_label': '10m', 'expected_blows': 60, 'position_basis': 'field_metadata'},
]

may19_assigned_frames = []
for spec in MAY19_CATALOG_GROUPS:
    start, end = pd.Timestamp(spec['start']), pd.Timestamp(spec['end'])
    d = may19_catalog.loc[may19_catalog.event_time.between(start, end, inclusive='both')].copy()
    d['candidate_origin'] = 'existing_unassigned_catalog'
    for key in ['group_key', 'assigned_source_x_m', 'assigned_source_label', 'expected_blows', 'position_basis']:
        d[key] = spec[key]
    may19_assigned_frames.append(d)

may19_catalog_assigned = pd.concat(may19_assigned_frames, ignore_index=True)
may19_recovered = recovered_events.loc[recovered_events.recovery_group_key.str.startswith('MAY19_')].copy()
may19_recovered['group_key'] = may19_recovered['recovery_group_key']
may19_assignments = pd.concat([may19_catalog_assigned, may19_recovered], ignore_index=True, sort=False)
may19_assignments['assignment_status'] = 'provisional_assigned'
may19_assignments['assignment_confidence'] = np.where(
    may19_assignments.group_key.eq('MAY19_SINKHOLE_110_114M'), 'medium', 'high'
)

may19_summary = (may19_assignments.groupby('group_key', as_index=False)
    .agg(assigned_source_x_m=('assigned_source_x_m', 'first'), assigned_source_label=('assigned_source_label', 'first'),
         expected_blows=('expected_blows', 'first'), n_candidates=('nodal_event_id', 'size'),
         first_event_utc=('event_time', 'min'), last_event_utc=('event_time', 'max'),
         candidate_origin=('candidate_origin', lambda s: ','.join(sorted(set(map(str, s))))),
         median_estimated_x_m=('estimated_source_x_m', 'median'))
)
may19_summary['count_minus_expected'] = may19_summary.n_candidates - may19_summary.expected_blows
display(may19_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)
summary_plot = may19_summary.sort_values('first_event_utc')
x = np.arange(len(summary_plot))
axes[0].bar(x - .18, summary_plot.expected_blows, width=.36, color='#999999', label='Expected blows')
axes[0].bar(x + .18, summary_plot.n_candidates, width=.36, color=COLORS['recovered'], label='Detected candidates')
axes[0].set_xticks(x, summary_plot.assigned_source_label, rotation=35, ha='right')
axes[0].set(title='May 19 expected versus detected candidates', ylabel='Count')
axes[0].legend()

for origin, d in may19_assignments.groupby('candidate_origin'):
    color = COLORS['catalog'] if origin == 'existing_unassigned_catalog' else COLORS['recovered']
    axes[1].scatter(d.event_time, d.assigned_source_x_m, s=18, color=color, alpha=.75, label=origin)
axes[1].axvline(pd.Timestamp('2026-05-19T13:20:00Z'), color='black', ls='--', lw=1, label='N2 → N3 geometry')
axes[1].set(title='May 19 provisional metadata-timed groups', xlabel='UTC time', ylabel='Assigned source x (m)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[1].legend(fontsize=8)
for ax in axes: ax.grid(alpha=.2)
fig.savefig(OUT_ROOT / '94b_may19_candidate_groups.png', dpi=180, bbox_inches='tight')
plt.close(fig)


## 4. May 17 duplicate suppression, outlier rejection and ordered segmentation

The segmentation assigns exactly 37 contiguous states from 104 through 140 m. It iteratively calibrates the nodal energy proxy to the ordered positions, uses a Huber loss so wild positions do not control boundaries, and discourages implausibly small or large groups. Excluded events remain visible in the rejection CSV and figure.

In [ ]:
def suppress_duplicate_triggers(frame, tolerance_s=DUPLICATE_TRIGGER_S):
    d = frame.sort_values('event_time').copy().reset_index(drop=True)
    d['rolling_x'] = d.estimated_source_x_m.rolling(21, center=True, min_periods=3).median()
    groups, current = [], [0]
    times = d.event_time.map(pd.Timestamp.timestamp).to_numpy(float)
    for i in range(1, len(d)):
        if times[i] - times[current[0]] < tolerance_s:
            current.append(i)
        else:
            groups.append(current); current = [i]
    if current: groups.append(current)
    keep, duplicate = [], []
    for group in groups:
        sub = d.loc[group]
        residual = (sub.estimated_source_x_m - sub.rolling_x).abs().fillna(0)
        chosen = residual.idxmin()
        keep.append(chosen)
        duplicate.extend([idx for idx in group if idx != chosen])
    kept = d.loc[keep].copy().sort_values('event_time')
    rejected = d.loc[duplicate].copy()
    rejected['provisional_rejection_reason'] = 'duplicate_trigger_within_0.8s'
    return kept, rejected

def huber(values, delta=2.0):
    a = np.abs(values)
    return np.where(a <= delta, .5 * a * a, delta * (a - .5 * delta))

def ordered_segmentation(frame, positions, min_len=5, max_len=45, iterations=3):
    d = frame.sort_values('event_time').reset_index(drop=True).copy()
    obs = d['smoothed_x'].to_numpy(float)
    n, k_total = len(d), len(positions)
    if n < k_total * min_len:
        raise RuntimeError(f'Only {n} usable events for {k_total} ordered groups')
    equal_state = np.minimum(k_total - 1, (np.arange(n) * k_total / n).astype(int))
    assigned = positions[equal_state]
    boundaries = None
    for _ in range(iterations):
        good = np.isfinite(obs) & np.isfinite(assigned)
        slope, intercept = np.polyfit(assigned[good], obs[good], 1)
        predicted = intercept + slope * positions
        target_len = n / k_total
        dp = np.full((k_total + 1, n + 1), np.inf)
        prev = np.full((k_total + 1, n + 1), -1, dtype=int)
        dp[0, 0] = 0.0
        for k in range(1, k_total + 1):
            lo_i = k * min_len
            hi_i = min(n, k * max_len)
            for i in range(lo_i, hi_i + 1):
                lo_j = max((k - 1) * min_len, i - max_len)
                hi_j = min((k - 1) * max_len, i - min_len)
                for j in range(lo_j, hi_j + 1):
                    if not np.isfinite(dp[k - 1, j]): continue
                    r = (obs[j:i] - predicted[k - 1]) / 2.5
                    fit_cost = np.nansum(huber(r, delta=2.0))
                    size_cost = 4.0 * ((i - j - target_len) / target_len) ** 2
                    cost = dp[k - 1, j] + fit_cost + size_cost
                    if cost < dp[k, i]:
                        dp[k, i], prev[k, i] = cost, j
        if not np.isfinite(dp[k_total, n]):
            raise RuntimeError('No feasible ordered segmentation; adjust group-size bounds')
        cuts = [n]
        i = n
        for k in range(k_total, 0, -1):
            i = prev[k, i]; cuts.append(i)
        cuts = list(reversed(cuts))
        state = np.empty(n, dtype=int)
        for k, (a, b) in enumerate(zip(cuts[:-1], cuts[1:])): state[a:b] = k
        assigned = positions[state]
        boundaries = cuts
    d['assigned_source_x_m'] = assigned
    d['group_key'] = [f'MAY17_{int(x):03d}M' for x in assigned]
    d['assigned_source_label'] = [f'{int(x)}m' for x in assigned]
    d['position_basis'] = 'ordered_104_to_140m_walk_constrained_segmentation'
    return d, {'slope': slope, 'intercept': intercept, 'boundaries': boundaries, 'cost': dp[k_total, n]}


In [ ]:
may17_existing = may17_catalog.copy()
may17_existing['candidate_origin'] = 'existing_unassigned_catalog'
may17_extension_all = recovered_events.loc[recovered_events.recovery_group_key.eq('MAY17_EXTENSION')].copy()
may17_extension = may17_extension_all.loc[may17_extension_all.event_time < MAY17_WALK_END].copy()
may17_late_recovery = may17_extension_all.loc[may17_extension_all.event_time >= MAY17_WALK_END].copy()
may17_late_recovery['provisional_rejection_reason'] = 'outside_pre_treefall_walk_window'
may17_combined = pd.concat([may17_existing, may17_extension], ignore_index=True, sort=False)
may17_deduped, may17_duplicates = suppress_duplicate_triggers(may17_combined)
may17_deduped = may17_deduped.sort_values('event_time').reset_index(drop=True)
may17_deduped['smoothed_x'] = may17_deduped.estimated_source_x_m.rolling(17, center=True, min_periods=5).median()
local_residual = (may17_deduped.estimated_source_x_m - may17_deduped.smoothed_x).abs()
outlier_mask = (
    (local_residual > MAY17_POSITION_OUTLIER_M)
    | (~may17_deduped.estimated_source_x_m.between(90, 155))
)
may17_outliers = may17_deduped.loc[outlier_mask].copy()
may17_outliers['provisional_rejection_reason'] = 'position_outlier_from_local_sequence'
may17_usable = may17_deduped.loc[~outlier_mask].copy()
may17_assignments, may17_fit = ordered_segmentation(
    may17_usable, MAY17_POSITIONS_M, min_len=MAY17_MIN_GROUP_EVENTS, max_len=MAY17_MAX_GROUP_EVENTS
)
may17_assignments['assignment_status'] = 'provisional_assigned'
may17_assignments['assignment_confidence'] = 'medium'
may17_expected_lookup = (
    may17_metadata.loc[may17_metadata.shot_no.between(1, 37)]
    .drop_duplicates('source_position_m').set_index('source_position_m')['n_blows']
)
may17_assignments['expected_blows'] = may17_assignments.assigned_source_x_m.map(may17_expected_lookup)
may17_rejected = pd.concat([may17_duplicates, may17_outliers, may17_late_recovery], ignore_index=True, sort=False)
may17_rejected['assignment_status'] = 'provisional_rejected_noise_or_duplicate'

may17_summary = (may17_assignments.groupby('group_key', as_index=False)
    .agg(assigned_source_x_m=('assigned_source_x_m', 'first'), expected_blows=('expected_blows', 'first'),
         n_candidates=('nodal_event_id', 'size'), first_event_utc=('event_time', 'min'), last_event_utc=('event_time', 'max'),
         median_estimated_x_m=('estimated_source_x_m', 'median'),
         n_recovered=('candidate_origin', lambda s: int((s == 'targeted_continuous_sds').sum())))
)
may17_summary['count_minus_expected'] = may17_summary.n_candidates - may17_summary.expected_blows
may17_summary['review_status'] = np.where(
    may17_summary.n_candidates.between(10, 30), 'plausible_count', 'review_count'
)
print('May 17 fit:', may17_fit)
print('Usable assigned:', len(may17_assignments), 'rejected/duplicates:', len(may17_rejected))
display(may17_summary)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), constrained_layout=True)
prewalk = may17_all_window.loc[may17_all_window.event_time < MAY17_CATALOG_START]
axes[0].scatter(prewalk.event_time, prewalk.estimated_source_x_m, s=8, color='#bbbbbb', alpha=.45, label='16:00–17:10 excluded/setup interval')
axes[0].scatter(may17_assignments.event_time, may17_assignments.estimated_source_x_m, s=9, color=COLORS['catalog'], alpha=.55, label='Usable sequence candidates')
axes[0].scatter(may17_rejected.event_time, may17_rejected.estimated_source_x_m, s=15, marker='x', color=COLORS['rejected'], alpha=.65, label='Duplicate/position outlier')
axes[0].set(title='May 17 raw nodal source proxy and exclusions', ylabel='Estimated source x (m)')
axes[0].legend(fontsize=8, ncol=3)

for origin, d in may17_assignments.groupby('candidate_origin'):
    color = COLORS['catalog'] if origin == 'existing_unassigned_catalog' else COLORS['recovered']
    axes[1].scatter(d.event_time, d.assigned_source_x_m, s=12, color=color, alpha=.7, label=origin)
axes[1].set(title='Provisional constrained 104–140 m staircase', ylabel='Assigned source x (m)')
axes[1].legend(fontsize=8)

x = np.arange(len(may17_summary))
axes[2].bar(x - .2, may17_summary.expected_blows, width=.4, color='#999999', label='Metadata blows')
axes[2].bar(x + .2, may17_summary.n_candidates, width=.4, color=COLORS['catalog'], label='Provisional candidates')
axes[2].set_xticks(x, may17_summary.assigned_source_x_m.astype(int), rotation=90)
axes[2].set(title='Per-position candidate counts', xlabel='Assigned source x (m)', ylabel='Count')
axes[2].legend()
for ax in axes[:2]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
for ax in axes: ax.grid(alpha=.2)
fig.savefig(OUT_ROOT / '94b_may17_ordered_walk_assignment.png', dpi=180, bbox_inches='tight')
plt.close(fig)


## 5. Export provisional assignments and review summaries

These files do not modify the catalog. The recovered continuous-data times must be extracted into full gathers before stacking.

In [ ]:
all_assignments = pd.concat([may17_assignments, may19_assignments], ignore_index=True, sort=False)
all_groups = pd.concat([
    may17_summary.assign(survey_date='2026-05-17', geometry='T1_N2'),
    may19_summary.assign(survey_date='2026-05-19', geometry=np.where(
        may19_summary.first_event_utc < pd.Timestamp('2026-05-19T13:20:00Z'), 'T1_N2', 'T1_N3'
    )),
], ignore_index=True, sort=False)

all_assignments.to_csv(OUT_ROOT / '94b_nodal_only_event_assignments_provisional.csv', index=False)
all_groups.to_csv(OUT_ROOT / '94b_nodal_only_shot_groups_provisional.csv', index=False)
may17_rejected.to_csv(OUT_ROOT / '94b_nodal_only_rejected_or_duplicate_events.csv', index=False)

run_summary = pd.DataFrame([
    ('May 17 original-window detections', len(may17_all_window)),
    ('May 17 forward-window catalog detections', len(may17_catalog)),
    ('May 17 targeted extension candidates through 19:30', len(may17_extension_all)),
    ('May 17 extension candidates used before 19:12', len(may17_extension)),
    ('May 17 provisionally assigned', len(may17_assignments)),
    ('May 17 provisional rejects/duplicates', len(may17_rejected)),
    ('May 19 catalog candidates assigned', len(may19_catalog_assigned)),
    ('May 19 targeted candidates recovered', len(may19_recovered)),
    ('Total provisional shot groups', len(all_groups)),
], columns=['metric', 'value'])
run_summary.to_csv(OUT_ROOT / '94b_run_summary.csv', index=False)
display(run_summary)
print('Review products:')
for path in sorted(OUT_ROOT.glob('94b_*')):
    print(' ', path)

## 6. Decision checkpoint

Before promotion or stacking:

1. Review all four targeted-recovery score panels. Regular trains are expected; isolated peaks are likely noise.
2. Review the May 17 staircase boundaries and per-position counts. Adjust the 17:10 start, extension threshold, or group-size bounds if needed.
3. Confirm whether the sinkhole-middle source should be recorded as 110, 114, or explicitly uncertain 110–114 m.
4. Extract full 3-component gathers for approved recovered events.
5. Apply notebook 95's consensus-reference waveform QC within every approved nodal-only group before stacking.
